# GMV 结构分解

GMV = 订单量 × 客单价。先把量价拆开，再按新客/老客拆，最后看品类结构变化。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
query("""
SELECT YEAR(purchase_ts) AS yr,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(price), 2) AS gmv,
       ROUND(SUM(price) / COUNT(DISTINCT order_id), 2) AS aov
FROM dws_order_wide
GROUP BY yr
ORDER BY yr
""")

,yr,orders,gmv,aov
0,2016,312,49785.92,159.57
1,2017,44579,6155806.98,138.09
2,2018,53775,7386050.80,137.35


## 新客与老客

每个客户的首单月份就是他的获客月，之后同一 unique_id 的订单算复购。

In [3]:
first_order = query("""
SELECT customer_unique_id, MIN(purchase_ts) AS first_ts
FROM dws_order_wide
GROUP BY customer_unique_id
""")

orders = query("""
SELECT order_id, customer_unique_id, purchase_ts, price
FROM dws_order_wide
GROUP BY order_id, customer_unique_id, purchase_ts, price
""")

orders = orders.merge(first_order, on="customer_unique_id")
orders["is_new"] = orders.purchase_ts == orders.first_ts
orders["month"] = orders.purchase_ts.dt.strftime("%Y-%m")

split = orders.groupby(["month", "is_new"]).price.sum().unstack(fill_value=0)
split.columns = ["复购", "新客首单"]
split["新客占比%"] = (split["新客首单"] / split.sum(axis=1) * 100).round(2)
split.tail(12)

,复购,新客首单,新客占比%
month,,,
2017-10,14320.43,600411.03,97.67
2017-11,22139.90,905720.87,97.61
2017-12,22156.10,681359.10,96.85
2018-01,24578.93,863490.61,97.23
2018-02,23585.19,757386.18,96.98
2018-03,22349.97,894206.17,97.56
2018-04,24733.84,908906.46,97.35
2018-05,30677.01,889126.42,96.66
2018-06,26332.09,779422.11,96.73


GMV 几乎全部由新客首单贡献，复购长期在 1-4% 之间，说明增长完全靠拉新，一旦投放收缩就会掉下来。